# Sprint 2 — André: Wide & Deep and Neighbour+Delta

Branch `s2/andre-models` (from `origin/ManuelS`). Trains four candidates on the dedup'd
training fold, scores them on the held-out validation fold, and reports each with
**exactly the team metrics** from `src/metrics.py`, following sections 8–9 of
`notebooks/sprint2_modeling.ipynb`. Two independent splits: `E.setup(seed=42)` and
`E.setup(seed=7)`.

| exp_id | model |
|---|---|
| `andre-champion-ref` | `BinaryRelevanceLGBM(min_child_samples=20, reg_lambda=1.0)`, threshold 0.5 |
| `andre-champion-rules42` | the same, then `data.apply_additive_rules` (the team champion) |
| `andre-wd512` | `WideDeepMultiLabel(hidden=(512,))` — `src/models_widedeep.py` |
| `andre-nndelta` | `NeighbourDelta()` — `src/models_nndelta.py` |
| `andre-blend-wd-nnd-champ` | equal average of the wd / nndelta / champion probabilities |

Team API only: `data.load()` (inside `E.setup`), `E.setup`, `E.dedup_configs`, `E.score`,
`E.log`, `E.Timer`, and `metrics as M` for every reported number — no hand-rolled metrics,
so the values line up with the team notebook.

## Blocker — `data.load()` cannot build its cache yet

`build_cache()` eagerly reads five CSVs; three are absent from `data/`:

| file | supplies | present |
|---|---|---|
| `train.csv` | `X_train`, `Y_raw` | yes |
| `test.csv` | `X_test` | yes |
| `train_clean.csv` | `Y_s10`, `s10_cols` | **no** |
| `train_clean_v2.csv` | `s11_cols`, `svd_train` | **no** |
| `test_clean_v2.csv` | `svd_test` | **no** |

This notebook needs **none** of those three — it uses `pipeline='raw'` (the 745 raw CRM
columns) and consensus targets derived from `Y_raw`. But `build_cache()` is all-or-nothing:
it writes a single `np.savez`, so it fails at `_header('train_clean.csv')` regardless. Fix by
either (a) obtaining the three Sprint 1 exports, (b) copying `data/cache/arrays.npz` from a
teammate, or (c) making the s10/s11 branches optional in `src/data.py` — which is shared code
and needs a PR. Until then the preflight cell below stops with a readable message.

## How to read these numbers

Scoring is on `E.setup(seed=...)` grouped holdouts — the Sprint 1 protocol, and the same one
the team notebook uses, which is why these numbers are comparable to it. Single-split
differences of well under ~1.5 points are not resolvable by two splits; the McNemar test at
the end is per **configuration** and is the only significance claim made here.
`solution.csv` is never read.

`andre-champion-rules42` is reported because it is the team's champion definition, but the 42
additive rules were mined on **all** of `train.csv`, so they leak validation labels. Its
margin over `andre-champion-ref` measures that leak; it is not a model improvement and must
not be used to choose a model.

## Setup

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from src import data, experiment as E, metrics as M
from src.models import BinaryRelevanceLGBM
from src.models_nndelta import NeighbourDelta
from src.models_widedeep import WideDeepMultiLabel

# data.load() (called inside E.setup) builds data/cache/arrays.npz from five Sprint 1 CSVs
# on first use, all-or-nothing. Fail with a readable message instead of a FileNotFoundError
# raised deep inside build_cache() after a 700 MB read.
_needed = ['train.csv', 'test.csv', 'train_clean.csv', 'train_clean_v2.csv', 'test_clean_v2.csv']
_missing = [f for f in _needed if not os.path.exists(os.path.join(data.DATA_DIR, f))]
if _missing and not os.path.exists(data.CACHE):
    raise FileNotFoundError(
        f"src/data.py needs {_missing} in data/ to build {data.CACHE}. Regenerate them with "
        "notebooks/sprint1_preprocessing_v3.ipynb, or copy data/cache/arrays.npz from a teammate.")

print('team loader cache :', data.CACHE, '| exists:', os.path.exists(data.CACHE))
print('team tracker      :', E.RESULTS)

team loader cache : c:\Users\andre\ECQ_Sprint2\ECQ\data\cache\arrays.npz | exists: True
team tracker      : c:\Users\andre\ECQ_Sprint2\ECQ\data\cache\results.jsonl


## Configuration

Model constructors take plain keyword arguments only, so Sprint 3 Optuna can rebuild them
from `trial.suggest_*`.

In [2]:
SEEDS = (42, 7)
VAL_SIZE = 0.2
THR = 0.5
PIPELINE = 'raw'                # the 745 raw CRM_* columns, i.e. data.features(d, 'raw')
RARE_MAX_TRAIN_ROWS = 200       # team convention: a CRM pack in < 200 training ROWS is rare
TOP_OFFENDERS = 10
MC_SEED = 42                    # seed used for the paired McNemar test
CMP_PATH = '../data/derived/sprint2_andre_comparison.csv'

CHAMP_PARAMS = dict(min_child_samples=20, reg_lambda=1.0)
WD_PARAMS = dict(hidden=(512,), dropout=0.2, wide=True, lr=2e-3,
                 batch_size=256, max_epochs=80, patience=8)
NND_PARAMS = dict(min_pairs=2, min_rate=0.5, max_nb=5)

EXP_CHAMP = 'andre-champion-ref'
EXP_CHAMP_RULES = 'andre-champion-rules42'
EXP_WD = 'andre-wd512'
EXP_NND = 'andre-nndelta'
EXP_BLEND = 'andre-blend-wd-nnd-champ'
ORDER = [EXP_CHAMP, EXP_CHAMP_RULES, EXP_WD, EXP_NND, EXP_BLEND]

# E.log appends to results.jsonl and does not enforce unique exp_ids, so the two seeds
# produce one record per (exp_id, split_seed), told apart by params['split_seed'].
pd.set_option('display.width', 160)

## One seed, end to end

`E.setup` gives the grouped holdout plus consensus targets; `E.dedup_configs` collapses the
training fold to unique CRM configurations. Its third return value is each configuration's
multiplicity, deliberately **unused** — training is unweighted on unique configurations.

Every metric is captured per model at fit time (`M.multilabel_report`, `M.row_errors`,
`M.top_offending_columns`) and only the small summaries are kept, so the two seeds never hold
five 731-column probability matrices at once.

`ok_cfg` records correctness on **unique validation configurations**: all rows of one CRM
configuration get identical predictions, so the configuration is the independent unit for the
paired McNemar test at the end.

In [3]:
def run_seed(split_seed):
    """Fit, score, log and fully profile every candidate for one holdout seed."""
    d = E.setup(seed=split_seed, val_size=VAL_SIZE)
    tr, va = d['tr'], d['va']
    X, Y_cons = d['X_train'], d['Y_cons']
    X_tr, Y_tr, _multiplicity = E.dedup_configs(X[tr], Y_cons[tr])
    X_va, Y_va = X[va], Y_cons[va]

    # one representative row per unique validation configuration
    _, cfg_first = np.unique(d['groups'][va], return_index=True)
    # pack frequency counted in training ROWS, as in the team notebook
    freq = X[tr].sum(0)
    rare = (freq > 0) & (freq < RARE_MAX_TRAIN_ROWS)
    has_rare = X_va[:, rare].any(1)

    print(f"split_seed {split_seed}: train {len(X_tr):,} unique configs (from {len(tr):,} rows) | "
          f"val {len(X_va):,} rows / {len(cfg_first):,} configs | {int(rare.sum())} rare packs")

    bundles = {}

    def record(exp_id, pred, proba, fit_s, family, params, notes):
        res = E.score(d, pred)
        E.log(exp_id, PIPELINE, family, dict(params, split_seed=split_seed), res, fit_s, notes=notes)
        summary, cm, f1 = M.multilabel_report(Y_va, pred, proba)
        err = M.row_errors(Y_va, pred)
        bundles[exp_id] = dict(
            summary=summary, cm=cm, f1=f1, err=err,
            offenders=M.top_offending_columns(Y_va, pred, d['bil_cols'], top_n=TOP_OFFENDERS),
            ok_cfg=(err[cfg_first] == 0), res=res)

    specs = [
        (EXP_CHAMP, 'lgbm-br', lambda: BinaryRelevanceLGBM(CHAMP_PARAMS),
         dict(CHAMP_PARAMS, model_seed=None),
         'BR LightGBM, mcs=20, reg_lambda=1, threshold 0.5, no rules'),
        (EXP_WD, 'widedeep', lambda: WideDeepMultiLabel(seed=split_seed, **WD_PARAMS),
         dict(WD_PARAMS, model_seed=split_seed),
         'wide additive path + shared 512-unit hidden layer'),
        (EXP_NND, 'nndelta', lambda: NeighbourDelta(**NND_PARAMS),
         dict(NND_PARAMS, model_seed=None),
         'nearest CRM config, then apply the per-pack BIL delta'),
    ]

    P = {}
    for exp_id, family, make, params, notes in specs:
        with E.Timer() as timer:
            model = make().fit(X_tr, Y_tr)
            P[exp_id] = np.asarray(model.predict_proba(X_va), dtype=np.float32)
        pred = (P[exp_id] >= THR).astype(np.uint8)
        record(exp_id, pred, P[exp_id], timer.s, family, params, notes)

        if exp_id == EXP_CHAMP:
            # the team's champion definition: overwrite the 42 deterministic BIL columns.
            # Those rules were mined on all of train.csv, so this LEAKS validation labels --
            # reported to quantify the leak, never to choose a model.
            record(EXP_CHAMP_RULES,
                   data.apply_additive_rules(pred, X_va, d['crm_cols'], d['bil_cols']),
                   P[exp_id], timer.s, family, params,
                   'champion + 42 Section 11 additive rules (LEAK: mined on all of train.csv)')

    members = [EXP_WD, EXP_NND, EXP_CHAMP]
    P_blend = np.mean([P[m] for m in members], axis=0)
    record(EXP_BLEND, (P_blend >= THR).astype(np.uint8), P_blend, 0.0, 'blend',
           dict(members=members, weights='equal', thr=THR),
           'equal average of the wd / nndelta / champion probabilities')
    del P, P_blend

    return dict(bundles=bundles, has_rare=has_rare, n_rare_packs=int(rare.sum()),
                rare_per_row_val=float(X_va[:, rare].sum(1).mean()),
                rare_per_row_test=float(d['X_test'][:, rare].sum(1).mean()),
                n_val_rows=len(va), n_val_cfg=len(cfg_first), n_train_cfg=len(X_tr))

## Run both seeds

Budget roughly 25–40 minutes per seed: the champion fits 731 LightGBM models (~3.5 min in the
team notebook), Wide & Deep runs up to 80 epochs, and each `M.multilabel_report` computes
micro ROC-AUC and PR-AUC over ~27 million cells. Each `E.log` line is appended as it
completes, so a later failure does not lose earlier records.

In [4]:
records = {}
for split_seed in SEEDS:
    print(f"\n{'=' * 78}\nsplit_seed {split_seed}\n{'=' * 78}")
    records[split_seed] = run_seed(split_seed)


split_seed 42
split_seed 42: train 53,947 unique configs (from 151,057 rows) | val 36,385 rows / 13,487 configs | 349 rare packs


c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-p

[andre-champion-ref] raw lgbm-br EMR=87.6543% HL=0.001327 d1=2.081% t=275s BR LightGBM, mcs=20, reg_lambda=1, threshold 0.5, no rules
[andre-champion-rules42] raw lgbm-br EMR=87.8411% HL=0.001375 d1=1.874% t=275s champion + 42 Section 11 additive rules (LEAK: mined on all of train.csv)
  ep   0 inner-EMR 0.5013 bce 0.02476 (4s) *
  ep   1 inner-EMR 0.5871 bce 0.02550 (7s) *
  ep   2 inner-EMR 0.6320 bce 0.02704 (11s) *
  ep   3 inner-EMR 0.6487 bce 0.02853 (15s) *
  ep   4 inner-EMR 0.6628 bce 0.02990 (18s) *
  ep   5 inner-EMR 0.6702 bce 0.03125 (22s) *
  ep   6 inner-EMR 0.6713 bce 0.03252 (26s) *
  ep   7 inner-EMR 0.6763 bce 0.03373 (29s) *
  ep   8 inner-EMR 0.6780 bce 0.03482 (33s) *
  ep   9 inner-EMR 0.6802 bce 0.03574 (36s) *
  ep  10 inner-EMR 0.6834 bce 0.03681 (40s) *
  ep  12 inner-EMR 0.6848 bce 0.03860 (48s) *
  ep  14 inner-EMR 0.6874 bce 0.04050 (55s) *
  ep  15 inner-EMR 0.6872 bce 0.04108 (59s)
  ep  17 inner-EMR 0.6893 bce 0.04278 (66s) *
  ep  20 inner-EMR 0.6911 b

c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\andre\anaconda3\Lib\site-p

[andre-champion-ref] raw lgbm-br EMR=86.3035% HL=0.001420 d1=2.692% t=495s BR LightGBM, mcs=20, reg_lambda=1, threshold 0.5, no rules
[andre-champion-rules42] raw lgbm-br EMR=86.4938% HL=0.001474 d1=2.478% t=495s champion + 42 Section 11 additive rules (LEAK: mined on all of train.csv)
  ep   0 inner-EMR 0.4705 bce 0.02433 (5s) *
  ep   1 inner-EMR 0.6016 bce 0.02495 (11s) *
  ep   2 inner-EMR 0.6305 bce 0.02651 (17s) *
  ep   3 inner-EMR 0.6479 bce 0.02795 (26s) *
  ep   4 inner-EMR 0.6641 bce 0.02930 (32s) *
  ep   5 inner-EMR 0.6685 bce 0.03057 (38s) *
  ep   6 inner-EMR 0.6759 bce 0.03182 (43s) *
  ep   7 inner-EMR 0.6787 bce 0.03303 (49s) *
  ep   9 inner-EMR 0.6796 bce 0.03505 (61s) *
  ep  10 inner-EMR 0.6837 bce 0.03602 (66s) *
  ep  11 inner-EMR 0.6839 bce 0.03686 (72s) *
  ep  12 inner-EMR 0.6841 bce 0.03786 (77s) *
  ep  13 inner-EMR 0.6869 bce 0.03867 (83s) *
  ep  14 inner-EMR 0.6871 bce 0.03945 (89s) *
  ep  15 inner-EMR 0.6874 bce 0.04028 (94s) *
  ep  17 inner-EMR 0.687

## Per-model report — the team metric set

For every model and seed, in the order of sections 8–9 of the team notebook:
`M.multilabel_report` summary (EMR, Hamming, micro/macro F1, precision, recall, AUCs), the
cell confusion matrix with false negatives per false positive, per-label F1
(median / F1 = 1 / F1 < 0.8), wrong bits per row (0…5, 6+), `M.top_offending_columns`, and
the rare-pack split.

In [5]:
for split_seed in SEEDS:
    rec = records[split_seed]
    for exp_id in ORDER:
        b = rec['bundles'][exp_id]
        summary, cm, f1, err = b['summary'], b['cm'], b['f1'], b['err']
        print(f"\n{'=' * 78}\n{exp_id}  |  split_seed {split_seed}\n{'=' * 78}")
        display(summary.to_frame(f'{exp_id} / seed {split_seed}').style.format('{:.5f}'))
        display(cm.style.format('{:,}'))
        print(f"false negatives per false positive: {cm.iloc[0, 1] / max(cm.iloc[1, 0], 1):.1f}")
        print(f"per-label F1 over {len(f1)} columns with positives: median {np.median(f1):.3f} | "
              f"F1 = 1: {(f1 == 1).sum()} | F1 < 0.8: {(f1 < 0.8).sum()}")
        print('wrong bits per validation row:',
              pd.Series(np.minimum(err, 6)).value_counts().sort_index()
                .rename(index={6: '6+'}).to_dict())
        print(f'\ncolumns that alone break an otherwise perfect row (top {TOP_OFFENDERS}):')
        display(b['offenders'].round(4))

        hr = rec['has_rare']
        display(pd.DataFrame({
            'val rows': [hr.sum(), (~hr).sum()],
            'EMR': [(err[hr] == 0).mean(), (err[~hr] == 0).mean()],
            'share with 6+ wrong bits': [(err[hr] >= 6).mean(), (err[~hr] >= 6).mean()],
        }, index=[f'contains a CRM pack with <{RARE_MAX_TRAIN_ROWS} train rows',
                  'all packs frequent']).round(4))

    print(f"\nseed {split_seed}: {rec['n_rare_packs']} rare CRM packs | mean per row: "
          f"validation {rec['rare_per_row_val']:.3f}, test {rec['rare_per_row_test']:.3f}")


andre-champion-ref  |  split_seed 42


,andre-champion-ref / seed 42
EMR,0.87654
bit accuracy,0.99867
Hamming loss,0.00133
F1 micro,0.96697
F1 samples,0.97367
F1 macro,0.37482
median per-label F1,0.22581
precision micro,0.99394
recall micro,0.94142
ROC-AUC micro,0.98572


,predicted 1,predicted 0
actual 1,"516,562","32,142"
actual 0,"3,151","26,045,580"


false negatives per false positive: 10.2
per-label F1 over 731 columns with positives: median 0.226 | F1 = 1: 0 | F1 < 0.8: 547
wrong bits per validation row: {0: 31893, 1: 757, 2: 313, 3: 220, 4: 238, 5: 253, '6+': 2711}

columns that alone break an otherwise perfect row (top 10):


,bil_column,n_rows_sole_error,missed_positive,false_positive,col_error_rate_all_rows,share_of_dist1_rows
0,BIL_SUBSCRIBER_STATUS_ORIG_Active,47,47,0,0.0050,0.0621
1,BIL_3940_PACK,44,36,8,0.0028,0.0581
2,BIL_4289_PACK,36,36,0,0.0021,0.0476
3,BIL_4297_PACK,31,31,0,0.0021,0.0410
4,BIL_3803_PACK,22,22,0,0.0022,0.0291
5,BIL_3812_PACK,16,16,0,0.0016,0.0211
6,BIL_3952_PACK,11,9,2,0.0021,0.0145
7,BIL_3855_PACK,11,11,0,0.0016,0.0145
8,BIL_3999_PACK,10,10,0,0.0019,0.0132
9,BIL_4156_PACK,10,10,0,0.0016,0.0132


...

,predicted 1,predicted 0
actual 1,"475,621","31,024"
actual 0,"4,150","23,682,381"


false negatives per false positive: 7.5
per-label F1 over 731 columns with positives: median 0.262 | F1 = 1: 0 | F1 < 0.8: 541
wrong bits per validation row: {0: 28891, 1: 523, 2: 243, 3: 227, 4: 226, 5: 272, '6+': 2714}

columns that alone break an otherwise perfect row (top 10):


,bil_column,n_rows_sole_error,missed_positive,false_positive,col_error_rate_all_rows,share_of_dist1_rows
0,BIL_3940_PACK,60,54,6,0.0036,0.1147
1,BIL_3803_PACK,25,25,0,0.0024,0.0478
2,BIL_4136_PACK,18,0,18,0.0015,0.0344
3,BIL_3762_PACK,11,1,10,0.0027,0.0210
4,BIL_4118_PACK,10,10,0,0.0013,0.0191
5,BIL_3801_PACK,10,10,0,0.0015,0.0191
6,BIL_4115_PACK,9,9,0,0.0015,0.0172
7,BIL_4102_PACK,9,9,0,0.0022,0.0172
8,BIL_SUBSCRIBER_STATUS_ORIG_Active,9,9,0,0.0036,0.0172
9,BIL_4311_PACK,8,8,0,0.0017,0.0153


,val rows,EMR,share with 6+ wrong bits
contains a CRM pack with <200 train rows,4157,0.1034,0.6375
all packs frequent,28939,0.9835,0.0022



seed 7: 355 rare CRM packs | mean per row: validation 0.480, test 0.039


## Comparison table

Rows = models, columns = (seed, metric) with the two seeds side by side. Saved to
`data/derived/sprint2_andre_comparison.csv`.

In [6]:
def summary_row(b):
    s, cm, err = b['summary'], b['cm'], b['err']
    return {
        'EMR': s['EMR'],
        'Hamming loss': s['Hamming loss'],
        'F1 micro': s['F1 micro'],
        'F1 macro': s['F1 macro'],
        'precision micro': s['precision micro'],
        'recall micro': s['recall micro'],
        'FN per FP': cm.iloc[0, 1] / max(cm.iloc[1, 0], 1),
        'median label F1': s['median per-label F1'],
        'share 6+ wrong bits': float((err >= 6).mean()),
    }


comparison = pd.concat(
    {f'seed {s}': pd.DataFrame({e: summary_row(records[s]['bundles'][e]) for e in ORDER}).T
     for s in SEEDS}, axis=1).reindex(ORDER)
comparison.index.name = 'exp_id'

os.makedirs(os.path.dirname(CMP_PATH), exist_ok=True)
comparison.to_csv(CMP_PATH)
print('written:', CMP_PATH, comparison.shape)
display(comparison.style.format('{:.5f}'))

written: ../data/derived/sprint2_andre_comparison.csv (5, 18)


## Paired exact McNemar test vs the champion, seed 42

Counted on **unique CRM configurations**, not rows: every row of a configuration gets the same
prediction, so a row-level test would treat thousands of identical rows as independent trials
and inflate significance. `b` and `c` are the discordant configurations; under the null they
split 50/50, which is an exact binomial test (`scipy.stats.binomtest`).

The reference is `andre-champion-ref` (no rules), so the comparison is leak-free. `EMR configs`
is per configuration and unweighted, so it differs from the row-weighted `EMR` above.

In [7]:
ref = records[MC_SEED]['bundles'][EXP_CHAMP]['ok_cfg']

rows = []
for exp_id in ORDER:
    if exp_id == EXP_CHAMP:
        continue
    a = records[MC_SEED]['bundles'][exp_id]['ok_cfg']
    only_a, only_ref = int((a & ~ref).sum()), int((~a & ref).sum())
    n_disc = only_a + only_ref
    rows.append({
        'exp_id': exp_id,
        'configs only this right (b)': only_a,
        'configs only champion right (c)': only_ref,
        'discordant': n_disc,
        'EMR configs': a.mean(),
        'dEMR configs vs champion': a.mean() - ref.mean(),
        'McNemar p': binomtest(only_a, n_disc, 0.5).pvalue if n_disc else 1.0,
    })

mcnemar = pd.DataFrame(rows).set_index('exp_id')
print(f"reference {EXP_CHAMP} | {len(ref):,} unique validation configurations | "
      f"champion EMR(configs) {ref.mean():.4f}")
display(mcnemar.style.format({'EMR configs': '{:.4f}',
                              'dEMR configs vs champion': '{:+.4f}',
                              'McNemar p': '{:.3g}'}))

reference andre-champion-ref | 13,487 unique validation configurations | champion EMR(configs) 0.6824


,configs only this right (b),configs only champion right (c),discordant,EMR configs,dEMR configs vs champion,McNemar p
exp_id,,,,,,
andre-champion-rules42,74,7,81,0.6873,+0.0050,3.17e-15
andre-wd512,256,150,406,0.6902,+0.0079,1.61e-07
andre-nndelta,301,536,837,0.6649,-0.0174,3.9e-16
andre-blend-wd-nnd-champ,247,56,303,0.6965,+0.0142,8.96e-30


## Notes

- **EMR is the competition metric; F1 and the AUCs are diagnostic only.** A row scores only if
  all 731 bits are right, so a model can improve micro F1 and still lose EMR. Read `EMR` and
  `share 6+ wrong bits` first; use `FN per FP` and `median label F1` to explain *why*.
- **`andre-champion-rules42` is not a candidate.** Its gap over `andre-champion-ref`
  quantifies the Sprint 1 rule leak (rules mined on all of `train.csv`), nothing more.
- **The rare-pack split is the live failure mode.** Validation rows containing a CRM pack seen
  in fewer than 200 training rows carry most of the 6+-wrong-bit rows, and the test set is
  richer in them — that gap is what Sprint 3 should target.
- **Two splits do not settle small margins.** Only the McNemar column supports a significance
  claim, and it covers seed 42 alone.
- `solution.csv` is not read anywhere in this notebook.

## Significance vs the team champion

Added after the run; the numbers live in `data/derived/sprint2_andre_significance.csv`.
The reference is `andre-champion-rules42`, which reproduces the team champion
`E5-reg-mcs20-t0.5` to six decimals on both seeds (0.878411 and 0.864938), so the
comparison is like-for-like on split, targets and recipe.

Two statistics, because they answer different questions:

* exact McNemar on the 13,487 **unique CRM configurations** — significance. Every row of a
  configuration receives the same prediction, so a row-level test would treat thousands of
  duplicates as independent trials and inflate significance.
* cluster (Poisson) bootstrap 95% CI on the **row-weighted** EMR difference — effect size in
  the units the leaderboard reports, resampling configurations rather than rows.

| model | seed 42 ΔEMR pts (95% CI) | seed 7 ΔEMR pts (95% CI) | verdict |
|---|---|---|---|
| blend | +0.42 (+0.13, +0.72) | +0.80 (+0.47, +1.23) | significant on both |
| blend + rules | +0.45 (+0.16, +0.74) | +0.82 (+0.49, +1.26) | significant on both |
| wd512 | +0.09 (−0.18, +0.31) | +0.49 (+0.15, +0.93) | **not established** |
| nndelta | −6.35 (−14.41, −1.25) | −1.07 (−1.86, −0.43) | significantly worse alone |

- **The blend is a genuine improvement** over the champion, on both splits, by both tests.
- **`wd512` alone is not.** Its seed-42 CI straddles zero even though McNemar returns
  p = 0.045 — it wins more configurations than it loses (200 vs 161), but the ones it loses
  carry more rows, so the effect on row-weighted EMR is indistinguishable from zero. Had only
  McNemar been run, this would have been reported as a win. Its value is inside the blend.
- **The 42 additive rules are nearly redundant once blended**: worth +0.19 EMR on the
  champion but only +0.022 on the blend. The ensemble already predicts those columns
  correctly. The clean result is therefore the blend *without* rules — the rules were mined
  on all of `train.csv`, validation rows included, so they carry a leak.
- `nndelta`'s very wide seed-42 interval (−14.41 to −1.25) is the quantitative form of its
  instability across splits: a few large configurations dominate its errors.

**Label correction.** These runs were logged with `pipeline='raw'` and relabelled afterwards
to `raw+consensus`, which is what the training actually was — `E.dedup_configs` on the train
fold with consensus targets, unweighted — and matches the team's taxonomy, where plain `raw`
means all rows with raw targets. The cell outputs above predate that correction.
